# Depth Fine-Tuning Through the Lens of PCA: DAv3 vs. DINOv2

[![Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lightly-ai/lightly-train/blob/main/examples/notebooks/depth_pca_visualization.ipynb)

[Depth Anything V3](https://depth-anything-v3.github.io/) (DAv3) is a monocular depth
estimation model built on a **DINOv2 ViT-L** backbone. During training, that backbone is
fine-tuned for the depth task. A natural question is: *how does depth fine-tuning change
the features the backbone produces?*

In this tutorial we answer that visually. We take the same trick as the
[ViT patch-feature PCA tutorial](https://colab.research.google.com/github/lightly-ai/lightly-train/blob/main/examples/notebooks/pca_visualization.ipynb)
— project each patch's feature vector onto its top three principal components and paint
them as RGB — and apply it to two backbones on the same image:

1. The **DINOv2 ViT-L encoder inside DAv3** (depth fine-tuned).
2. The **original DINOv2 ViT-L** (self-supervised, no depth training).

Because both are the *exact same architecture* (ViT-L/14, 1024-dim features, no register
tokens), the comparison is apples-to-apples — the only difference is the weights.

## Install Dependencies

We only need `lightly-train`, which brings `torch`, `numpy`, and `matplotlib` along with
it. The PCA is computed directly with `torch`.

In [ ]:
!pip install lightly-train

## Get a Sample Image

Let's download a single image with clear foreground/background structure — depth features
tend to reorganize most visibly along depth discontinuities. Feel free to swap in your
own.

In [ ]:
import io
import urllib.request

from PIL import Image

url = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
image = Image.open(io.BytesIO(urllib.request.urlopen(url).read())).convert("RGB")
image

## Preprocessing and PCA Helpers

Both backbones are DINOv2 ViT-L/14 with ImageNet normalization, so they share one
preprocessing function. We resize to **504×504** — the resolution DAv3 runs at, and a
multiple of the patch size 14 (giving a 36×36 patch grid).

The `pca_rgb` helper is the same one from the ViT patch-feature PCA tutorial: center the
patch features, project onto their top three principal components with
`torch.pca_lowrank`, and percentile-normalize the result to RGB.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms.v2 as transforms

# DINOv2 ViT-L/14 details (shared by DAv3's encoder and the original DINOv2).
PATCH_SIZE = 14
IMAGE_SIZE = 896  # DAv3's inference resolution; a multiple of the patch size.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def preprocess(image):
    transform = transforms.Compose(
        [
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToImage(),
            transforms.ToDtype(torch.float32, scale=True),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ]
    )
    return transform(image).unsqueeze(0)  # (1, 3, H, W)


def extract_features(vit, x):
    # Last-layer patch tokens as a spatial grid (1, feature_dim, H, W).
    out = vit.get_intermediate_layers(
        x, n=1, reshape=True, return_class_token=False, norm=True
    )
    return out[0]


def pca_rgb(features):
    """Maps ViT patch features (1, feature_dim, H, W) to an (H, W, 3) RGB image."""
    _, feature_dim, h, w = features.shape
    tokens = features[0].permute(1, 2, 0).reshape(-1, feature_dim)
    tokens = tokens - tokens.mean(dim=0, keepdim=True)
    _, _, v = torch.pca_lowrank(tokens, q=3)
    projected = (tokens @ v[:, :3]).reshape(h, w, 3).cpu().numpy()
    low = np.percentile(projected, 1, axis=(0, 1), keepdims=True)
    high = np.percentile(projected, 99, axis=(0, 1), keepdims=True)
    return np.clip((projected - low) / (high - low + 1e-8), 0, 1)


def upsample_to(rgb, size):
    grid = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0)
    grid = F.interpolate(grid, size=size, mode="bilinear", align_corners=False)
    return grid[0].permute(1, 2, 0).numpy()


x = preprocess(image)
print(f"input shape: {tuple(x.shape)}")

## Load DAv3 and Grab Its DINOv2 Encoder

We load the hosted DAv3 relative-depth model with `lightly_train.load_model`. The model's
DINOv2 ViT-L encoder is exposed as `model.backbone`, a standard `DinoVisionTransformer` we
can extract patch features from directly.

In [ ]:
import lightly_train

# Load the DAv3 relative-depth model (weights download on first use).
dav3 = lightly_train.load_model("dinov2/dav3-relative-large", device="cpu")

# Its DINOv2 ViT-L encoder (depth fine-tuned).
dav3_encoder = dav3.backbone.eval()
with torch.no_grad():
    dav3_features = extract_features(dav3_encoder, x)

print(f"encoder:      {type(dav3_encoder).__name__}")
print(f"feature dim:  {dav3_encoder.embed_dim}")
print(f"feature grid: {tuple(dav3_features.shape)}")

## Load the Original DINOv2 ViT-L

For a fair baseline we load the matching original DINOv2 ViT-L. We use the **`-noreg`**
(no register token) variant because that is exactly the architecture DAv3 fine-tuned, so
the two only differ in their weights.

```{note}
`get_wrapped_model` lives under `lightly_train._models` and is an internal API that may
change between releases. We use it here to load a raw backbone; `.get_model()` returns the
underlying `DinoVisionTransformer` with the same interface as `dav3.backbone`.
```

In [ ]:
from lightly_train._models.package_helpers import get_wrapped_model

# Original self-supervised DINOv2 ViT-L/14 (no register tokens, to match DAv3).
dinov2_encoder = (
    get_wrapped_model("dinov2/vitl14-noreg", num_input_channels=3).get_model().eval()
)
with torch.no_grad():
    dinov2_features = extract_features(dinov2_encoder, x)

print(f"feature grid: {tuple(dinov2_features.shape)}")
assert dinov2_features.shape == dav3_features.shape

## Compare the Feature Maps

Now we run PCA on each backbone's features and plot them side by side. Each panel computes
its own PCA basis, so the specific colors are not comparable between panels — what to look
for is the *structure*: how the features group the scene, and where the boundaries fall.

In [ ]:
import matplotlib.pyplot as plt

panels = [
    ("Input", None),
    ("DAv3 encoder (depth fine-tuned)", dav3_features),
    ("Original DINOv2 ViT-L", dinov2_features),
]

fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
for ax, (title, features) in zip(axes, panels):
    if features is None:
        ax.imshow(image.resize((IMAGE_SIZE, IMAGE_SIZE)))
    else:
        ax.imshow(upsample_to(pca_rgb(features), IMAGE_SIZE))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Conclusion

By visualizing the same ViT-L backbone before and after depth fine-tuning, we can see how
the DAv3 training reshapes DINOv2's features: the depth-tuned encoder tends to organize
patches more strongly by scene geometry and depth ordering, while the original DINOv2
features reflect its self-supervised, appearance-driven objective. Both remain the same
architecture, so the difference is entirely in what the weights have learned.

This is a quick, label-free way to sanity-check what a fine-tuned foundation model is
paying attention to. You can try the `dinov2/dav3-metric-large` model (metric depth), swap
in your own images, or explore the full
[depth estimation workflow](https://docs.lightly.ai/train/stable/depth_estimation.html)
in the LightlyTrain documentation.